In [ ]:
%%capture
!pip install keras-tuner tensorflow scikit-learn matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD
import keras_tuner as kt


# Breast Cancer Dataset

In [ ]:
# Carregando o Breast Cancer
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Split e padronização
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Função para o Breast Cancer
def construir_modelo_cancer(hp):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    units = hp.Int('units', min_value=4, max_value=64, step=4)
    model.add(Dense(units, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    momentum = hp.Float('momentum', min_value=0.0, max_value=0.9, step=0.1)
    optimizer = SGD(learning_rate=lr, momentum=momentum)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

tuner_cancer = kt.RandomSearch(
    construir_modelo_cancer,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='tuning_cancer',
    project_name='cancer_classification'
)

tuner_cancer.search(X_train, y_train, epochs=100, validation_split=0.2, verbose=0)

melhor_modelo_cancer = tuner_cancer.get_best_models(1)[0]
melhores_hp_cancer = tuner_cancer.get_best_hyperparameters(1)[0]

# Avaliação
loss, acc = melhor_modelo_cancer.evaluate(X_test, y_test, verbose=0)
print(f"Acurácia Breast Cancer: {acc:.2f}")
print("Hiperparâmetros Breast Cancer:", melhores_hp_cancer.values)

# Relatório
y_pred_test = (melhor_modelo_cancer.predict(X_test) > 0.5).astype("int32")
print("\nClassification Report (Breast Cancer):")
print(classification_report(y_test, y_pred_test))


# Wine Dataset

In [ ]:
# Carregando o Wine
wine = load_wine()
X = wine.data
y = wine.target

# Split e padronização
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Função para o Wine (multiclasse)
def construir_modelo_wine(hp):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    units = hp.Int('units', min_value=4, max_value=64, step=4)
    model.add(Dense(units, activation='relu'))
    model.add(Dense(3, activation='softmax'))
    lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    optimizer = SGD(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

tuner_wine = kt.RandomSearch(
    construir_modelo_wine,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='tuning_wine',
    project_name='wine_classification'
)

tuner_wine.search(X_train, y_train, epochs=100, validation_split=0.2, verbose=0)

melhor_modelo_wine = tuner_wine.get_best_models(1)[0]
melhores_hp_wine = tuner_wine.get_best_hyperparameters(1)[0]

# Avaliação
loss, acc = melhor_modelo_wine.evaluate(X_test, y_test, verbose=0)
print(f"Acurácia Wine: {acc:.2f}")
print("Hiperparâmetros Wine:", melhores_hp_wine.values)

# Relatório
y_pred_test = np.argmax(melhor_modelo_wine.predict(X_test), axis=1)
print("\nClassification Report (Wine):")
print(classification_report(y_test, y_pred_test))
